In [0]:
books_pd = spark.table("workspace.default.books").toPandas()
books_pd[["book_id", "title", "genres"]].head(10)

In [0]:
print(type(books_pd["genres"].iloc[0]))
print(books_pd["genres"].iloc[0])

In [0]:
import ast

books_pd["genres"] = books_pd["genres"].apply(lambda x: ast.literal_eval(x))
print(type(books_pd["genres"].iloc[0]))

In [0]:
books_pd["genred_text"] = books_pd["genres"].apply(lambda x: " ".join(x))
books_pd[["title", "genres", "genred_text"]].head(3)

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer 

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(books_pd["genred_text"])
print(tfidf_matrix.shape)

In [0]:
#compare every book's 45 nhumber row against every other book to see similar score 
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)
print(similarity_matrix.shape)

In [0]:
#use for real recomandations
#for one book, we want to find the books with the highest similiary score to it
def recommend_similar_books(title, n=5):
    idx = books_pd[books_pd["title"] == title].index[0] #find row nr for the given title
    scores = list(enumerate(similarity_matrix[idx])) #get book's similiary scores against other book
    scores = sorted(scores, key=lambda x: x[1], reverse=True) #sort by similiarity score, highest first
    
    top_matches = [(i, s) for i, s in scores if i != idx][:n] #skip first result (book itself) and take the next n
    recommended_indices = [i for i, score in top_matches] #look up the actual titles for those row numbers
    
    return books_pd.iloc[recommended_indices][["title", "genres"]]


In [0]:
recommend_similar_books("The Hobbit")

In [0]:
ratings_pd = spark.table("workspace.default.ratings").toPandas()
print(ratings_pd["user_id"].nunique())
print(ratings_pd["user_id"].max())
print(ratings_pd["book_id"].nunique())
print(ratings_pd["book_id"].max())

In [0]:
from scipy.sparse import csr_matrix

user_item_matrix = csr_matrix((ratings_pd["rating"], (ratings_pd["user_id"] - 1, ratings_pd["book_id"] - 1)))
print(user_item_matrix.shape)

In [0]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=20, random_state=42)
book_factors = svd.fit_transform(user_item_matrix.T)

print(book_factors.shape)

In [0]:
from sklearn.metrics.pairwise import cosine_similarity

collab_similarity_matrix = cosine_similarity(book_factors)
print(collab_similarity_matrix.shape)

In [0]:
def recommend_collaborative(title, n=5):
    idx = books_pd[books_pd["title"] == title].index[0]
    scores = list(enumerate(collab_similarity_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_matches = [(i, s) for i, s in scores if i != idx][:n]
    recommended_indices = [i for i, score in top_matches]
    return books_pd.iloc[recommended_indices][["title", "genres"]]

In [0]:
recommend_collaborative("The Hobbit")

In [0]:
liked = ratings_pd[ratings_pd["rating"] >= 4]
user_like_counts = liked.groupby("user_id").size()
eligible_users = user_like_counts[user_like_counts >= 2].index
print(len(eligible_users))

In [0]:
%pip install scikit-surprise

In [0]:
%restart_python

In [0]:
%pip install numpy cython

In [0]:
%restart_python

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [0]:
books_pd = spark.read.table("workspace.default.books")
display(books_pd)

In [0]:
ratings_pd = spark.read.table("workspace.default.ratings")
display(ratings_pd)